# 💳 Credit Card Fraud Detection using XGBoost

This notebook demonstrates the development of a machine learning model for credit card fraud detection using:

* Class imbalance handling
* Bayesian hyperparameter optimization
* XGBoost classification
* Threshold optimization using Precision-Recall analysis

The objective is to build and evaluate a robust fraud detection model for identifying fraudulent transactions.



## 1. Imports & Setup


In [1]:

import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_recall_curve
from bayes_opt import BayesianOptimization



## 2. Load Dataset

> Dataset is not included due to size.
> Download `creditcard.csv` from Kaggle and place it in the project root.


In [3]:

df = pd.read_csv("creditcard.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

print("Dataset shape:", df.shape)
print(y.value_counts())


Dataset shape: (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64



## 3. Train-Test Split & Class Imbalance Handling


In [4]:

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print("Scale pos weight:", scale_pos_weight)


Scale pos weight: 577.2868020304569



## 4. Baseline XGBoost Model


In [5]:

baseline_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="mlogloss"
)

baseline_model.fit(x_train, y_train)
y_pred_base = baseline_model.predict(x_test)

print("Baseline Model")
print(confusion_matrix(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base))


Baseline Model
[[56863     1]
 [   17    81]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.99      0.83      0.90        98

    accuracy                           1.00     56962
   macro avg       0.99      0.91      0.95     56962
weighted avg       1.00      1.00      1.00     56962




## 5. Bayesian Optimization for Hyperparameter Tuning


In [8]:

def xgb_evaluate(max_depth, learning_rate, n_estimators, scale_pos_weight):
    model = XGBClassifier(
        max_depth=int(max_depth),
        learning_rate=learning_rate,
        n_estimators=int(n_estimators),
        scale_pos_weight=scale_pos_weight,
        eval_metric="mlogloss"
    )
    
    scores = cross_val_score(
        model, x_train, y_train,
        scoring="f1", cv=5, n_jobs=-1
    )
    return scores.mean()


In [9]:

optimizer = BayesianOptimization(
    f=xgb_evaluate,
    pbounds={
        "max_depth": (2, 6),
        "learning_rate": (0.01, 0.3),
        "n_estimators": (50, 150),
        "scale_pos_weight": (1, scale_pos_weight)
    },
    random_state=42
)

optimizer.maximize(init_points=5, n_iter=10)


|   iter    |  target   | max_depth | learni... | n_esti... | scale_... |
-------------------------------------------------------------------------
| 1         | 0.7999291 | 3.4981604 | 0.2857071 | 123.19939 | 345.99898 |
| 2         | 0.1589647 | 2.6240745 | 0.0552384 | 55.808361 | 500.16588 |
| 3         | 0.6085142 | 4.4044600 | 0.2153410 | 52.058449 | 559.94624 |
| 4         | 0.7612965 | 5.3297705 | 0.0715783 | 68.182496 | 106.69359 |
| 5         | 0.6610232 | 3.2169689 | 0.1621793 | 93.194501 | 168.83150 |
| 6         | 0.5255891 | 2.6868673 | 0.2680345 | 122.06733 | 345.84805 |
| 7         | 0.4969011 | 3.8028952 | 0.1981715 | 65.363762 | 438.57430 |
| 8         | 0.5033488 | 3.8381389 | 0.0686854 | 59.053230 | 182.55296 |
| 9         | 0.7548620 | 4.0760248 | 0.0450972 | 57.839764 | 59.068012 |
| 10        | 0.5176358 | 4.1011689 | 0.0562672 | 132.60930 | 429.42926 |
| 11        | 0.8624584 | 5.7960730 | 0.2942758 | 110.89563 | 288.01128 |
| 12        | 0.5920686 | 3.0194995 | 


## 6. Train Final Tuned Model


In [11]:

best_params = optimizer.max["params"]

final_model = XGBClassifier(
    max_depth=int(best_params["max_depth"]),
    learning_rate=best_params["learning_rate"],
    n_estimators=int(best_params["n_estimators"]),
    scale_pos_weight=best_params["scale_pos_weight"],
    eval_metric="mlogloss"
)

final_model.fit(x_train,y_train)

y_pred_final = final_model.predict(x_test)

print("Tuned Model")
print(confusion_matrix(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))


Tuned Model
[[56860     4]
 [   20    78]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.95      0.80      0.87        98

    accuracy                           1.00     56962
   macro avg       0.98      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962




## 7. Business-Oriented Threshold Selection


In [34]:
y_probs = final_model.predict_proba(x_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
    if r >= 0.83 and p >= 0.95:     
        print("Threshold:", t)
        print("Precision:", p)
        print("Recall:", r)
        break

Threshold: 0.33306038
Precision: 0.9534883720930233
Recall: 0.8367346938775511


## 8. Final Evaluation at Selected Threshold

In [35]:
y_pred = (y_probs > 0.33306038).astype(int)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.95      0.83      0.89        98

    accuracy                           1.00     56962
   macro avg       0.98      0.91      0.94     56962
weighted avg       1.00      1.00      1.00     56962




## 9. Key Takeaways

* Accuracy can be misleading in fraud detection tasks.
* Effective handling of class imbalance is critical.
* Bayesian Optimization enables efficient hyperparameter tuning.
* Optimized thresholds help balance fraud detection and false positives.
